In [29]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna

from dysts.maps import Henon

# Init

## Init Reservoir

In [30]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [31]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [32]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [33]:
u_val = 1
u_dist = 400
u_dataset = u_val * np.sin(2 * np.pi * t / (2 * u_dist))
u_dataset = u_dataset[transient_steps_chaos:]

## Init Funcs

In [34]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [35]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [36]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [37]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    combined_weights = weights.reshape(2, -1).T.flatten()

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [38]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [39]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [40]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    init_vecs = (
        initial_pos[connections_list[:, 0]] - initial_pos[connections_list[:, 1]]
    )
    rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Lin Bayesian

In [24]:
N = 10

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [25]:
tau_steps = 1
free_steps = 3
dt = 0.01

target_nodes = np.array([3, 8])
wall_nodes = [0]

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [26]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, -1)

connections_list = np.column_stack((src_nodes, dst_nodes))

In [ ]:
force_data = henon_scaled
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [ ]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha):
    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()

    if np.any(np.isnan(X)):
        return (), (-1.0, 1e9), ()

    X_delayed = X[: -tau_steps * free_steps]
    X_data = X_delayed[transient_steps_reservoir * free_steps :]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :].repeat(
        free_steps, axis=0
    )

    rest_test_steps = test_steps * free_steps
    X_train, X_test = (
        X_data[:-rest_test_steps],
        X_data[-rest_test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-rest_test_steps],
        Y_data[-rest_test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [29]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    2, 0.3, 10, 4, 0.1
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.3146 0.4218


In [19]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, timeout=15, n_jobs=-1)

[I 2026-07-21 15:47:04,798] A new study created in memory with name: no-name-630fad14-12fc-4ea6-bec8-fea6f73c36d8


[Optuna] Processing Trial #26...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #104...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


...

[Optuna] Processing Trial #236...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #282...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real
/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #285...

In [20]:
for trial in study.best_trials[:10]:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #216
  Values: [0.6118352620185606, 0.3326625728659436]
  Params: {'mass': 0.0147440549876883, 'damping': 0.48907510247545827, 'stiffness': 7.390792023262226, 'input_force': 9.432877102853036, 'ridge_alpha': 0.0012338546821270028}
Trial #228
  Values: [0.6107998418055713, 0.3263512978151559]
  Params: {'mass': 0.023984553997877398, 'damping': 2.6667736573391516, 'stiffness': 3.965072410436379, 'input_force': 1.9379962886856095, 'ridge_alpha': 0.0012338546821270028}


In [21]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"],
)

print(r_2, mse)

0.6118352620185606 0.3326625728659436


In [22]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

2026-07-21 15:47:32.600 python[91439:7363938] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-21 15:47:32.600 python[91439:7363938] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [ ]:
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Non Lin Bayesian Multiple

In [83]:
N = 10
dist_between = 2.5

x = np.zeros(N*3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 1, 0]) + dist_between*i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [84]:
tau_steps = 1
dt = 0.01

target_nodes = np.array([4, -5])
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [85]:
node_ids = np.arange(x.size)

# 0->1 1->2 then 3->4 4->5 then 6->7 7->8
# 0, 1, 3, 4, 6, 7
# 1, 2, 4, 5, 7, 8
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()

# Between Connections
# 1->4 then 4->7
betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]

src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])

connections_list = np.column_stack((src_nodes, dst_nodes))

In [ ]:
def bayesian_trial(m_val, c_val, k_val, k_between_val, input_force, ridge_alpha, free_steps):
    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = np.ones(connection_src.shape[0]) * k_val
    k_between_vals = np.ones(between_src.shape[0]) * k_between_val
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    force_data = henon_scaled
    total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)
    U = np.zeros((total_steps_with_free, matrix_size))
    col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
    col_indices = col_indices[::2]
    ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
    U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
        :, : col_indices.shape[0]
    ]

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()
    if not np.isfinite(X).all():
        return (), (-1.0, 1e9), ()

    X_sampled = X[free_steps - 1 :: free_steps]
    X_delayed = X_sampled[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return v

In [145]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    .3, 0.3, 10, 1, 40, 0.1, 20
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.4657 0.3863


In [71]:
displacement[:, 8].min(), displacement[:, 8].max()

(np.float64(-2.6992277269208076), np.float64(0.910452109055716))

In [146]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    0.01, 0.3, 10, 1, 10, 0.1, 1
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6280 0.3182


In [74]:
displacement[:, 8].min(), displacement[:, 8].max()

(np.float64(-0.8241340615040195), np.float64(0.5594525240548134))

In [147]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    0.01, 0.3, .1, 1, 10, 0.1, 15
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6936 0.2843


In [139]:
N = 20
results = np.zeros((N-1, 3))

for i in range(1, N):
    (Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
        0.01, 0.3, 0.1, 1, 10, 0.1, i
    )

    results[i-1, 0] = i
    results[i-1, 1] = r_2
    results[i-1, 2] = mse

In [140]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    template="plotly_dark",
)

fig.update_yaxes(title_text="R² Score", secondary_y=False)
fig.update_yaxes(title_text="MSE", secondary_y=True)

fig.show()

In [148]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    0.01, 0.3, 0.1, .3, 10, 0.1, 15
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6644 0.2975


In [149]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
).show()

In [193]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.001, 10, log=True),
        trial.suggest_float("damping", 0.001, 10, log=True),
        trial.suggest_float("stiffness_conn", 0.001, 10, log=True),
        trial.suggest_float("stiffness_between", 0.001, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
        3
    )

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)

[Optuna] Processing Trial #32...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #47...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #63...

...

[Optuna] Processing Trial #956...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #964...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #999...

In [195]:
for trial in study.best_trials[:10]:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #911
  Values: [0.6914554229637973, 0.28599664103166406]
  Params: {'mass': 0.0010582839166430856, 'damping': 0.23955106660909775, 'stiffness_conn': 1.3088546396681175, 'stiffness_between': 7.318720309894253, 'input_force': 8.463862474492819, 'ridge_alpha': 0.0014263055577193354}


In [197]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness_conn"],
    params["stiffness_between"],
    params["input_force"],
    params["ridge_alpha"],
    3
)

print(r_2, mse)

0.6914554229637973 0.28599664103166406


In [202]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
).show()

In [199]:
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Non Lin Bayesian Multiple

In [42]:
def spring_trial(
    N, target_nodes, m_val, c_val, k_val, k_between_val, input_force, ridge_alpha, free_steps
):
    dist_between = 2.5
    x = np.zeros(N*3)
    for i in range(0, N):
        x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between*i
    y = np.tile(np.array([0, 1, 2]), N)
    nodes_pos = np.column_stack((x, y))

    tau_steps = 1
    dt = 0.01
    starts = 3 * np.arange(N)
    wall_nodes = np.column_stack([starts, starts + 2]).flatten()
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]
    matrix_size = num_nodes * dims

    node_ids = np.arange(x.size)

    starts = 3 * np.arange(N)
    connection_src = np.column_stack([starts, starts + 1]).flatten()
    connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()
    betweens = np.arange(1, node_ids[-1], 3)
    between_src = betweens[:-1]
    between_dst = betweens[1:]
    src_nodes = np.concatenate([connection_src, between_src])
    dst_nodes = np.concatenate([connection_dst, between_dst])
    connections_list = np.column_stack((src_nodes, dst_nodes))

    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = np.ones(connection_src.shape[0]) * k_val
    k_between_vals = np.ones(between_src.shape[0]) * k_between_val
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)
    U = np.zeros((total_steps_with_free, matrix_size))
    for i, node_index in enumerate(target_nodes):
        # We only do X
        U[::free_steps, node_index * dims] = henon_scaled[:, i % dims]

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()
    if not np.isfinite(X).all():
        return (), (-1.0, 1e9), ()

    X_sampled = X[free_steps - 1 :: free_steps]
    X_delayed = X_sampled[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity), (nodes_pos, connections_list, model)

In [44]:
target_nodes = np.array([4, -5])
_, (r_2, mse), _, _ = spring_trial(10, target_nodes, 0.3, 0.3, 10, 1, 40, 0.1, 20)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6139 0.3283


In [45]:
target_nodes = np.array([4, -5])
_, (r_2, mse), _, _ = spring_trial(10, target_nodes, 0.01, 0.3, 10, 1, 10, 0.1, 1)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6332 0.3140


In [46]:
target_nodes = np.array([4, -5])
_, (r_2, mse), _, _ = spring_trial(10, target_nodes, 0.01, 0.3, 0.1, 1, 10, 0.1, 15)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6301 0.3123


In [48]:
free_vals = np.arange(1, 20) * 3
results = np.zeros((len(free_vals), 3))

for i, free_val in enumerate(free_vals):
    target_nodes = np.array([4, -5])
    _, (r_2, mse), _, _ = spring_trial(
        10, target_nodes, 0.01, 0.3, 0.1, 1, 10, 0.1, free_val
    )

    results[i, 0] = free_val
    results[i, 1] = r_2
    results[i, 2] = mse

In [49]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    template="plotly_dark",
)

fig.update_yaxes(title_text="R² Score", secondary_y=False)
fig.update_yaxes(title_text="MSE", secondary_y=True)

fig.show()

In [274]:
target_nodes = np.array([1])
(
    (Y_test, Y_pred),
    (r_2, mse),
    (displacement, velocity),
    (nodes_pos, connections_list),
) = spring_trial(
    N=1,
    target_nodes=target_nodes,
    m_val=0.01,
    c_val=0.3,
    k_val=.3,
    k_between_val=0.3,
    input_force=10,
    ridge_alpha=0.1,
    free_steps=11,
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.5380 0.3633


In [16]:
N = 300
N_step = int(N / 5)
target_nodes = 1 + np.array([N_step, 2 * N_step, 3 * N_step, 4 * N_step]) * 3
(
    (Y_test, Y_pred),
    (r_2, mse),
    (displacement, velocity),
    (nodes_pos, connections_list),
) = spring_trial(
    N=N,
    target_nodes=target_nodes,
    m_val=0.01,
    c_val=0.3,
    k_val=0.3,
    k_between_val=0.3,
    input_force=10,
    ridge_alpha=0.1,
    free_steps=11,
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6320 0.3116


In [291]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
).show()